# Tanzania Financial News Scraper
### DS & AI Year I: Beautiful Soup Tutotial

This notebook collects financial news headlines from major Tanzanian news outlets
for the period **2022 – 2024**. The collected dataset will be used for:

- Sentiment analysis of financial news
- Correlation analysis with TZS/USD exchange rates and inflation data

**Target outlets:**
- Daily News Tanzania (`dailynews.co.tz`)
- The Citizen Tanzania (`thecitizen.co.tz`)

**Authors:** *(add your names)*  
**Date:** *(add date)*

---

> **Before running:** Make sure you have installed the required libraries.  
> Run the cell in **Section 1** first if this is your first time.


## Section 1: Requirements & Imports

Run the install cell once if you don't have the libraries yet.  
After that you only need to run the imports cell every session.


In [ ]:
# ── Install once (comment out after first run) ──
# !pip install requests beautifulsoup4

In [2]:
# ── Imports (run every session) ──
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

print("All libraries loaded successfully")

All libraries loaded successfully


## Section 2: How BeautifulSoup Works

Before scraping real websites, let's understand the core tool we're using:
**BeautifulSoup (bs4)**.

A webpage is just HTML, a tree of nested tags. BeautifulSoup lets us
navigate and extract from that tree using CSS selectors, tag names, or attributes.


### a.) Parsing hardcoded HTML (no internet needed)

Below is a tiny fake news page. We'll extract headlines and dates from it
exactly the same way we will from real sites.

In [3]:
# Fake HTML that looks like a news listing page
fake_html = """
<html>
  <body>
    <div class="news-list">

      <div class="article">
        <h2 class="headline">
          <a href="Thek2news/article/1">Shilling strengthens against the dollar</a>
        </h2>
        <span class="date">2023-04-12</span>
      </div>

      <div class="article">
        <h2 class="headline">
          <a href="Thek2news/article/2">Inflation drops to 3.8 percent in March</a>
        </h2>
        <span class="date">2023-04-10</span>
      </div>

      <div class="article">
        <h2 class="headline">
          <a href="Thek2news/article/3">Tanzania secures new Chinese FDI deal</a>
        </h2>
        <span class="date">2023-04-08</span>
      </div>

    </div>
  </body>
</html>
"""

# Parse it
soup = BeautifulSoup(fake_html, 'html.parser')

# Extract all articles
articles = soup.select('div.article')
print(f"Found {len(articles)} articles\n")

for article in articles:
    headline = article.select_one('h2.headline a').get_text(strip=True)
    date     = article.select_one('span.date').get_text(strip=True)
    link     = article.select_one('h2.headline a')['href']
    print(f" {date} | {headline}")
    print(f" Link: {link}\n")


Found 3 articles

 2023-04-12 | Shilling strengthens against the dollar
 Link: Thek2news/article/1

 2023-04-10 | Inflation drops to 3.8 percent in March
 Link: Thek2news/article/2

 2023-04-08 | Tanzania secures new Chinese FDI deal
 Link: Thek2news/article/3



**What just happened?**

| Code | What it does |
|------|-------------|
| `BeautifulSoup(html, 'html.parser')` | Parses the raw HTML string into a navigable tree |
| `soup.select('div.article')` | Finds ALL elements matching the CSS selector |
| `soup.select_one('h2.headline a')` | Finds the FIRST matching element |
| `.get_text(strip=True)` | Extracts the visible text, removing whitespace |
| `['href']` | Reads an HTML attribute value |

This is the entire foundation of web scraping everything else is just applying
these same ideas to real websites.


### b.) Scraping a live website

Now the same idea but on a real URL. We use `requests` to download the page HTML,
then BeautifulSoup to parse it.

We use **`books.toscrape.com`** — a site built specifically for scraping practice.
It has no login, no blocks, and a clean predictable structure. Perfect for learning.


In [9]:
# Download the page
url = "http://books.toscrape.com/"
response = requests.get(url, timeout=10) # timeout is how long the fetxh is going to keep ontrying to receive the site if exceeded returns error

print(f"Status code : {response.status_code}")   # 200 = success
print(f"Content size: {len(response.text):,} characters of HTML")


Status code : 200
Content size: 51,294 characters of HTML


In [10]:
# Parse and extract book titles + prices
soup = BeautifulSoup(response.text, 'html.parser')

books = soup.select('article.product_pod')
print(f"Found {len(books)} books on this page\n")

for book in books[:5]:   # show first 5 only
    title = book.select_one('h3 a')['title']
    price = book.select_one('p.price_color').get_text(strip=True)
    print(f" Book Name:  {title}")
    print(f" Book Price: {price}\n")


Found 20 books on this page

 Book name:  A Light in the Attic
 Book Price: Â£51.77

 Book name:  Tipping the Velvet
 Book Price: Â£53.74

 Book name:  Soumission
 Book Price: Â£50.10

 Book name:  Sharp Objects
 Book Price: Â£47.82

 Book name:  Sapiens: A Brief History of Humankind
 Book Price: Â£54.23



---
### Understanding the Web Scraping Code

`BeautifulSoup(...)` parses the HTML page.

```python
soup.select('article.product_pod')
```

This finds all `<article>` elements with class `product_pod`, where each represents a book.

So `books` becomes a list of all book elements.

---

##### Looping Through Books

```python
books[:5]
```

This limits the loop to only the first **5 books**.

Useful for:

* Testing
* Previewing results
* Avoiding excessive output

---

##### Extracting the Title

```python
title = book.select_one('h3 a')['title']
```

This:

* Finds the `<a>` tag inside `<h3>`
* Reads its `title` attribute

Example:

```html
<a title="A Light in the Attic">
```

Result:

```python
title = "A Light in the Attic"
```

---

##### Extracting the Price

```python
price = book.select_one('p.price_color').get_text(strip=True)
```

This finds the price element and extracts its text.

Example:

```html
<p class="price_color">£53.74</p>
```

Result:

```python
price = "£53.74"
```

The scraper now extracts both the **book title** and **price**.


**Key difference from the hardcoded demo:**

We added one step — `requests.get(url)` — which downloads the raw HTML from the internet.
After that, BeautifulSoup works exactly the same way.

This is the complete scraping workflow:
```
URL → requests.get() → raw HTML → BeautifulSoup() → extract data → save
```

Everything in the rest of this notebook is just this loop, applied to Tanzanian news sites.
